# PCA robustness analyses


In [ ]:
from pathlib import Path
import os
import warnings
import numpy as np
import pandas as pd
from scipy.optimize import linear_sum_assignment
from sklearn.decomposition import PCA
RESULTS_ROOT = Path(os.environ.get('EMOTION_RESULTS_ROOT', 'results'))
INPUT_FILE = Path(
    os.environ.get(
        'EMOTION_MAIN_SUMMARY_FILE',
        str(
            RESULTS_ROOT / '1_reliability_description' / 'results_files' / 'reliability_description_summary.xlsx'
        )
    )
)
OUTPUT_DIR = Path(
    os.environ.get('EMOTION_ROBUSTNESS_OUTPUT_DIR', str(RESULTS_ROOT / '3_pca_robustness_analyses'))
)
PROCESS_DIR = OUTPUT_DIR / 'process_files'
RESULTS_DIR = OUTPUT_DIR / 'results_files'
for folder in [OUTPUT_DIR, PROCESS_DIR, RESULTS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)
N_COMPONENTS = 5
N_BOOTSTRAP = int(os.environ.get('EMOTION_N_BOOTSTRAP', '1000'))
BOOTSTRAP_RANDOM_SEED = 2026
RELIABILITY_THRESHOLD = 0.7
N_PERMUTATIONS = int(os.environ.get('EMOTION_N_PERMUTATIONS', '10000'))
PARALLEL_RANDOM_SEED = 2027
PARALLEL_PERCENTILE = 95
if not INPUT_FILE.exists():
    raise FileNotFoundError(f'Cannot find input file: {INPUT_FILE}')


In [ ]:
def z_standardize_columns(data):
    array = np.asarray(data, dtype=float)
    means = array.mean(axis=0)
    sds = array.std(axis=0, ddof=1)
    if np.any(~np.isfinite(sds)) or np.any(sds <= 0):
        raise ValueError('At least one feature has a zero or invalid sample SD.')
    return ((array - means) / sds, means, sds)

def fit_pca(standardized_data, n_components):
    model = PCA(n_components=n_components, svd_solver='full')
    scores = model.fit_transform(standardized_data)
    eigenvalues = model.explained_variance_.copy()
    ratios = model.explained_variance_ratio_.copy()
    loadings = model.components_.T * np.sqrt(eigenvalues)
    return (model, scores, loadings, eigenvalues, ratios)

def orient_by_largest_loading(loadings, scores=None, feature_names=None):
    oriented_loadings = np.asarray(loadings, dtype=float).copy()
    oriented_scores = None if scores is None else np.asarray(scores, dtype=float).copy()
    rows = []
    for component_index in range(oriented_loadings.shape[1]):
        anchor_index = int(np.argmax(np.abs(oriented_loadings[:, component_index])))
        before = float(oriented_loadings[anchor_index, component_index])
        sign = 1.0 if before >= 0 else -1.0
        oriented_loadings[:, component_index] *= sign
        if oriented_scores is not None:
            oriented_scores[:, component_index] *= sign
        rows.append(
            {
                'Component': f'PC{component_index + 1}',
                'Anchor Feature': feature_names[anchor_index] if feature_names is not None else anchor_index,
                'Loading Before Orientation': before,
                'Sign Multiplier': sign,
                'Loading After Orientation': float(oriented_loadings[anchor_index, component_index])
            }
        )
    return (oriented_loadings, oriented_scores, pd.DataFrame(rows))

def tucker_congruence(vector_a, vector_b):
    a = np.asarray(vector_a, dtype=float)
    b = np.asarray(vector_b, dtype=float)
    denominator = np.sqrt(np.sum(a ** 2) * np.sum(b ** 2))
    if denominator <= 0 or not np.isfinite(denominator):
        return np.nan
    return float(np.sum(a * b) / denominator)

def pairwise_tucker(reference_loadings, candidate_loadings):
    reference = np.asarray(reference_loadings, dtype=float)
    candidate = np.asarray(candidate_loadings, dtype=float)
    output = np.empty((reference.shape[1], candidate.shape[1]), dtype=float)
    for reference_index in range(reference.shape[1]):
        for candidate_index in range(candidate.shape[1]):
            output[reference_index, candidate_index] = tucker_congruence(
                reference[:, reference_index],
                candidate[:, candidate_index]
            )
    return output

def match_components(reference_loadings, candidate_loadings):
    pairwise = pairwise_tucker(reference_loadings, candidate_loadings)
    reference_indices, candidate_indices = linear_sum_assignment(-np.abs(pairwise))
    order = np.argsort(reference_indices)
    candidate_indices = candidate_indices[order]
    signed_values = np.array(
        [
            pairwise[reference_index, candidate_index] for reference_index,
            candidate_index in enumerate(candidate_indices)
        ]
    )
    signs = np.where(signed_values >= 0, 1.0, -1.0)
    matched_loadings = candidate_loadings[:, candidate_indices] * signs
    congruence = np.abs(signed_values)
    return (matched_loadings, candidate_indices, signs, congruence, pairwise)

def parallel_analysis(standardized_data, n_permutations, percentile, seed):
    X = np.asarray(standardized_data, dtype=float)
    n_rows, n_features = X.shape
    n_components = min(n_rows, n_features)
    observed = np.linalg.eigvalsh(np.corrcoef(X, rowvar=False))[::-1]
    rng = np.random.default_rng(seed)
    permuted = np.empty_like(X)
    null_eigenvalues = np.empty((n_permutations, n_components), dtype=float)
    for iteration in range(n_permutations):
        for feature_index in range(n_features):
            permuted[:, feature_index] = rng.permutation(X[:, feature_index])
        null_eigenvalues[iteration] = np.linalg.eigvalsh(np.corrcoef(permuted, rowvar=False))[::-1]
    null_threshold = np.percentile(null_eigenvalues, percentile, axis=0)
    above = observed > null_threshold
    retained = 0
    for flag in above:
        if bool(flag):
            retained += 1
        else:
            break
    labels = [f'PC{index}' for index in range(1, n_components + 1)]
    summary = pd.DataFrame(
        {
            'Component': labels,
            'Observed Eigenvalue': observed,
            'Null Mean': null_eigenvalues.mean(axis=0),
            'Null SD': null_eigenvalues.std(axis=0, ddof=1),
            'Null 2.5th Percentile': np.percentile(null_eigenvalues, 2.5, axis=0),
            'Null Median': np.percentile(null_eigenvalues, 50, axis=0),
            f'Null {percentile}th Percentile': null_threshold,
            'Null 97.5th Percentile': np.percentile(null_eigenvalues, 97.5, axis=0),
            f'Observed Minus Null {percentile}th': observed - null_threshold,
            f'Observed Above Null {percentile}th': above,
            'Retained Sequentially': [index < retained for index in range(n_components)]
        }
    )
    return (summary, null_eigenvalues, retained)


In [ ]:
mean_matrix_raw = pd.read_excel(INPUT_FILE, sheet_name='Mean_Rating_Matrix')
reliability_table = pd.read_excel(INPUT_FILE, sheet_name='Split_Half_Reliability')
required_columns = {'Emotion Feature', 'Mean Split-Half Reliability'}
if 'Emotion Word' not in mean_matrix_raw.columns:
    raise ValueError("Mean_Rating_Matrix must contain an 'Emotion Word' column.")
if not required_columns.issubset(reliability_table.columns):
    raise ValueError(f'Split_Half_Reliability must contain: {sorted(required_columns)}')
emotion_words = mean_matrix_raw['Emotion Word'].astype('string').str.strip()
feature_names = [column for column in mean_matrix_raw.columns if column != 'Emotion Word']
mean_matrix = mean_matrix_raw[feature_names].apply(pd.to_numeric, errors='coerce')
if len(emotion_words) != 70 or len(feature_names) != 43:
    raise ValueError(
        f'Expected a 70 x 43 matrix, found {len(emotion_words)} x {len(feature_names)}.'
    )
if emotion_words.isna().any() or emotion_words.duplicated().any():
    raise ValueError('Emotion words contain missing or duplicate values.')
if mean_matrix.isna().any().any():
    raise ValueError('The mean-rating matrix contains missing or nonnumeric values.')
reliability_table['Emotion Feature'] = reliability_table['Emotion Feature'].astype('string').str.strip()
reliability_lookup = reliability_table.set_index('Emotion Feature')['Mean Split-Half Reliability'].astype(float)
if set(feature_names) != set(reliability_lookup.index):
    raise ValueError('Feature names differ between the mean matrix and reliability table.')
X_raw = mean_matrix.to_numpy(dtype=float)
X, _, _ = z_standardize_columns(X_raw)
n_words, n_features = X.shape
_, primary_scores_raw, primary_loadings_raw, primary_eigenvalues, primary_ratios = fit_pca(X, N_COMPONENTS)
primary_loadings, primary_scores, primary_orientation = orient_by_largest_loading(
    primary_loadings_raw,
    primary_scores_raw,
    feature_names
)
component_labels = [f'PC{index}' for index in range(1, N_COMPONENTS + 1)]
primary_loadings_df = pd.DataFrame(primary_loadings, index=feature_names, columns=component_labels)
primary_loadings_df.index.name = 'Emotion Feature'
primary_scores_df = pd.DataFrame(primary_scores, index=emotion_words, columns=component_labels)
primary_scores_df.index.name = 'Emotion Word'
primary_component_summary = pd.DataFrame(
    {
        'Component': component_labels,
        'Eigenvalue': primary_eigenvalues,
        'Explained Variance Percentage': primary_ratios * 100,
        'Cumulative Variance Percentage': np.cumsum(primary_ratios) * 100
    }
)
primary_loadings_df.to_csv(RESULTS_DIR / '01_primary_component_loadings.csv', encoding='utf-8-sig')
primary_scores_df.to_csv(RESULTS_DIR / '02_primary_emotion_word_scores.csv', encoding='utf-8-sig')


In [ ]:
rng = np.random.default_rng(BOOTSTRAP_RANDOM_SEED)
bootstrap_indices = np.empty((N_BOOTSTRAP, n_words), dtype=np.int16)
bootstrap_congruence = np.empty((N_BOOTSTRAP, N_COMPONENTS), dtype=float)
bootstrap_eigenvalues = np.empty((N_BOOTSTRAP, N_COMPONENTS), dtype=float)
bootstrap_component_order = np.empty((N_BOOTSTRAP, N_COMPONENTS), dtype=np.int8)
bootstrap_signs = np.empty((N_BOOTSTRAP, N_COMPONENTS), dtype=np.int8)
bootstrap_matched_loadings = np.empty((N_BOOTSTRAP, n_features, N_COMPONENTS), dtype=np.float32)
for iteration in range(N_BOOTSTRAP):
    sampled_indices = rng.integers(0, n_words, size=n_words)
    bootstrap_indices[iteration] = sampled_indices
    X_bootstrap, _, _ = z_standardize_columns(X_raw[sampled_indices, :])
    _, _, candidate_loadings, candidate_eigenvalues, _ = fit_pca(X_bootstrap, N_COMPONENTS)
    matched, candidate_order, signs, congruence, _ = match_components(primary_loadings, candidate_loadings)
    bootstrap_congruence[iteration] = congruence
    bootstrap_eigenvalues[iteration] = candidate_eigenvalues[candidate_order]
    bootstrap_component_order[iteration] = candidate_order + 1
    bootstrap_signs[iteration] = signs.astype(np.int8)
    bootstrap_matched_loadings[iteration] = matched.astype(np.float32)
bootstrap_summary = pd.DataFrame(
    {
        'Component': component_labels,
        'Mean Tucker Congruence': bootstrap_congruence.mean(axis=0),
        'Median Tucker Congruence': np.median(bootstrap_congruence, axis=0),
        'Tucker Congruence SD': bootstrap_congruence.std(axis=0, ddof=1),
        '2.5th Percentile': np.percentile(bootstrap_congruence, 2.5, axis=0),
        '97.5th Percentile': np.percentile(bootstrap_congruence, 97.5, axis=0),
        'Minimum': bootstrap_congruence.min(axis=0),
        'Maximum': bootstrap_congruence.max(axis=0),
        'Proportion >= .95': (bootstrap_congruence >= 0.95).mean(axis=0),
        'Bootstrap Iterations': N_BOOTSTRAP
    }
)
bootstrap_iterations = pd.DataFrame(bootstrap_congruence, columns=component_labels)
bootstrap_iterations.insert(0, 'Bootstrap Iteration', np.arange(1, N_BOOTSTRAP + 1))
bootstrap_mapping_rows = []
for iteration in range(N_BOOTSTRAP):
    for reference_index, component in enumerate(component_labels):
        bootstrap_mapping_rows.append(
            {
                'Bootstrap Iteration': iteration + 1,
                'Reference Component': component,
                'Bootstrap Native Component': f'PC{
                    bootstrap_component_order[iteration, reference_index]
                }',
                'Sign Multiplier': int(bootstrap_signs[iteration, reference_index]),
                'Tucker Congruence': float(bootstrap_congruence[iteration, reference_index]),
                'Bootstrap Eigenvalue': float(bootstrap_eigenvalues[iteration, reference_index])
            }
        )
bootstrap_mapping = pd.DataFrame(bootstrap_mapping_rows)
loading_stability_rows = []
aligned_float = bootstrap_matched_loadings.astype(float)
for component_index, component in enumerate(component_labels):
    for feature_index, feature in enumerate(feature_names):
        values = aligned_float[:, feature_index, component_index]
        primary_value = float(primary_loadings[feature_index, component_index])
        primary_sign = 1 if primary_value >= 0 else -1
        lower = float(np.percentile(values, 2.5))
        upper = float(np.percentile(values, 97.5))
        loading_stability_rows.append(
            {
                'Component': component,
                'Emotion Feature': feature,
                'Primary Loading': primary_value,
                'Bootstrap Mean Loading': float(np.mean(values)),
                'Bootstrap Loading SD': float(np.std(values, ddof=1)),
                'Bootstrap 2.5th Percentile': lower,
                'Bootstrap 97.5th Percentile': upper,
                'Proportion With Primary Sign': float(np.mean(np.sign(values) == primary_sign)),
                '95% Interval Excludes Zero': bool(lower > 0 or upper < 0)
            }
        )
bootstrap_loading_stability = pd.DataFrame(loading_stability_rows)
bootstrap_iterations.to_csv(
    PROCESS_DIR / '01_bootstrap_congruence_iterations.csv',
    index=False,
    encoding='utf-8-sig'
)
bootstrap_mapping.to_csv(
    PROCESS_DIR / '02_bootstrap_component_matching.csv',
    index=False,
    encoding='utf-8-sig'
)
bootstrap_loading_stability.to_csv(
    PROCESS_DIR / '03_bootstrap_loading_stability.csv',
    index=False,
    encoding='utf-8-sig'
)
np.savez_compressed(
    PROCESS_DIR / '04_bootstrap_complete_arrays.npz',
    sampled_word_indices=bootstrap_indices,
    tucker_congruence=bootstrap_congruence,
    matched_native_components=bootstrap_component_order,
    sign_multipliers=bootstrap_signs,
    matched_eigenvalues=bootstrap_eigenvalues,
    matched_loadings=bootstrap_matched_loadings
)
bootstrap_summary.to_csv(
    RESULTS_DIR / '03_bootstrap_component_stability_summary.csv',
    index=False,
    encoding='utf-8-sig'
)


In [ ]:
low_reliability_features = [
    feature for feature in feature_names if float(reliability_lookup.loc[feature]) < RELIABILITY_THRESHOLD
]
retained_feature_names = [
    feature for feature in feature_names if feature not in low_reliability_features
]
removed_features = pd.DataFrame(
    {
        'Emotion Feature': low_reliability_features,
        'Mean Split-Half Reliability': [
            float(reliability_lookup.loc[feature]) for feature in low_reliability_features
        ],
        'Exclusion Threshold': RELIABILITY_THRESHOLD
    }
)
if not low_reliability_features:
    warnings.warn('No feature fell below the reliability threshold.')
reduced_raw = mean_matrix[retained_feature_names].to_numpy(dtype=float)
reduced_X, reduced_means, reduced_sds = z_standardize_columns(reduced_raw)
reduced_parallel, reduced_null, reduced_n_retained = parallel_analysis(
    reduced_X,
    N_PERMUTATIONS,
    PARALLEL_PERCENTILE,
    PARALLEL_RANDOM_SEED
)
if reduced_n_retained < N_COMPONENTS:
    raise ValueError(
        f'Reduced parallel analysis retained only {reduced_n_retained} components; cannot compare {N_COMPONENTS} components.'
    )
_, reduced_scores_native, reduced_loadings_native, reduced_eigenvalues, reduced_ratios = fit_pca(reduced_X, N_COMPONENTS)
common_indices = [feature_names.index(feature) for feature in retained_feature_names]
primary_common = primary_loadings[common_indices, :]
reduced_loadings_matched, reduced_order, reduced_signs, reduced_congruence, reduced_pairwise = match_components(primary_common, reduced_loadings_native)
reduced_scores_matched = reduced_scores_native[:, reduced_order] * reduced_signs
reduced_mapping = pd.DataFrame(
    {
        'Primary Component': component_labels,
        'Reduced Native Component': [f'PC{index + 1}' for index in reduced_order],
        'Sign Multiplier': reduced_signs,
        'Tucker Congruence': reduced_congruence,
        'Reduced Eigenvalue': reduced_eigenvalues[reduced_order],
        'Reduced Explained Variance Percentage': reduced_ratios[reduced_order] * 100
    }
)
reduced_native_summary = pd.DataFrame(
    {
        'Reduced Native Component': component_labels,
        'Eigenvalue': reduced_eigenvalues,
        'Explained Variance Percentage': reduced_ratios * 100,
        'Cumulative Variance Percentage': np.cumsum(reduced_ratios) * 100
    }
)
reduced_loadings_df = pd.DataFrame(
    reduced_loadings_matched,
    index=retained_feature_names,
    columns=component_labels
)
reduced_loadings_df.index.name = 'Emotion Feature'
reduced_scores_df = pd.DataFrame(
    reduced_scores_matched,
    index=emotion_words,
    columns=component_labels
)
reduced_scores_df.index.name = 'Emotion Word'
reduced_pairwise_df = pd.DataFrame(
    reduced_pairwise,
    index=component_labels,
    columns=[f'Reduced_PC{index}' for index in range(1, N_COMPONENTS + 1)]
)
reduced_pairwise_df.index.name = 'Primary Component'
reduced_null_df = pd.DataFrame(
    reduced_null,
    columns=[f'PC{index}' for index in range(1, reduced_null.shape[1] + 1)]
)
reduced_null_df.insert(0, 'Permutation', np.arange(1, N_PERMUTATIONS + 1))
reduced_null_df.to_csv(
    PROCESS_DIR / '05_reduced_parallel_permutation_eigenvalues.csv',
    index=False,
    encoding='utf-8-sig'
)
removed_features.to_csv(
    RESULTS_DIR / '04_removed_low_reliability_features.csv',
    index=False,
    encoding='utf-8-sig'
)
reduced_parallel.to_csv(
    RESULTS_DIR / '05_reduced_parallel_analysis_summary.csv',
    index=False,
    encoding='utf-8-sig'
)
reduced_mapping.to_csv(
    RESULTS_DIR / '06_reduced_component_congruence.csv',
    index=False,
    encoding='utf-8-sig'
)
reduced_loadings_df.to_csv(RESULTS_DIR / '07_reduced_component_loadings.csv', encoding='utf-8-sig')
reduced_scores_df.to_csv(RESULTS_DIR / '08_reduced_emotion_word_scores.csv', encoding='utf-8-sig')


In [ ]:
summary_workbook = RESULTS_DIR / 'pca_robustness_analyses_summary.xlsx'
with pd.ExcelWriter(summary_workbook, engine='openpyxl') as writer:
    primary_component_summary.to_excel(writer, sheet_name='Primary_Component_Stats', index=False)
    primary_loadings_df.reset_index().to_excel(writer, sheet_name='Primary_Loadings', index=False)
    bootstrap_summary.to_excel(writer, sheet_name='Bootstrap_Summary', index=False)
    bootstrap_iterations.to_excel(writer, sheet_name='Bootstrap_Iterations', index=False)
    bootstrap_loading_stability.to_excel(
        writer,
        sheet_name='Bootstrap_Loading_Stability',
        index=False
    )
    removed_features.to_excel(writer, sheet_name='Removed_Features', index=False)
    reduced_parallel.to_excel(writer, sheet_name='Reduced_Parallel', index=False)
    reduced_native_summary.to_excel(writer, sheet_name='Reduced_Component_Stats', index=False)
    reduced_mapping.to_excel(writer, sheet_name='Reduced_Congruence', index=False)
    reduced_pairwise_df.reset_index().to_excel(
        writer,
        sheet_name='Reduced_Pairwise_Tucker',
        index=False
    )
    reduced_loadings_df.reset_index().to_excel(writer, sheet_name='Reduced_Loadings', index=False)
    reduced_scores_df.reset_index().to_excel(writer, sheet_name='Reduced_Word_Scores', index=False)
